In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# from matplotlib.colors import LogNorm
# from glob import glob
import os
from scipy.stats import linregress, t
plt.rcParams.update({"text.usetex": True,})
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from params import *

In [ ]:
def load_HPLC(fname, params):
    
    HPLC_df = pd.read_csv(fname ,sep='\t',parse_dates=[['Date','Time_UTC']],index_col=0)
    HPLC_df = HPLC_df.drop(columns=['Proben ID','Event_label','jultime','depth','Cryptophytes','Chrysophytes'])
    HPLC_df = HPLC_df.rename_axis('time').sort_index()
    HPLC_df = HPLC_df.rename(columns={'Lat':'lat','Lon':'lon'})
    HPLC_df = HPLC_df.rename(columns=params['PFT_HPLC_dict'])
    HPLC_df['PROKAR'] = HPLC_df['PROKAR'] + HPLC_df['PROCHLO']
    HPLC_df = HPLC_df.drop(columns=['PROCHLO'])
    HPLC = HPLC_df.to_xarray().set_coords(['lat','lon'])
    
    
    for var_name in HPLC.data_vars:
        HPLC[var_name] = HPLC[var_name].assign_attrs(units=params['units'])
        HPLC[var_name] = HPLC[var_name].assign_attrs(long_name=params['PFT_longname'][var_name])

    HPLC['time'] = HPLC.time.astype('datetime64[m]')
    HPLC = HPLC.where(np.isfinite(HPLC))
    return HPLC

In [ ]:
def load_ACS(fname, params):
    ACS = xr.open_dataset(fname).compute()
    ACS = ACS.rename({'Time':'time','Lat':'lat','Lon':'lon'})
    ACS = ACS.rename(params['PFT_ACS_dict'])
    return ACS

In [ ]:
HPLC = load_HPLC("/Users/emehdipo/awi/ACS/matchup/HPLC_data_pigments_PFT_all_PS113_dpa_Alvarado2021_MChla.txt", params)

In [ ]:
# ACS = load_ACS("/Users/emehdipo/awi/ACS/data/1min/PFT_concentration.nc", params)
ACS = load_ACS("/Users/emehdipo/awi/ACS/data/4s/PFT_concentration_log.nc", params)
ACS_std = load_ACS("/Users/emehdipo/awi/ACS/data/4s/PFT_concentrations_log_std.nc", params)

In [ ]:
ACS = 10**ACS

# Functions

In [ ]:
def plot_prediction_band(ax, x, y, slope, intercept, color='r'):
    """Plots the 95% prediction band for log-log regression from 0.001 to 10."""
    log_x = np.log10(x)
    log_y = np.log10(y)

    # Define x values for prediction, ensuring full range coverage
    x_vals = np.logspace(np.log10(0.001), np.log10(10), 100)
    log_x_pred = np.log10(x_vals)
    log_y_pred = slope * log_x_pred + intercept
    y_vals = 10**log_y_pred

    # Compute residual standard error
    residuals = log_y - (slope * log_x + intercept)
    residual_std_err = np.std(residuals, ddof=2)

    # Compute prediction interval
    n = len(log_x)
    t_value = t.ppf(0.975, df=n-2)  # 95% confidence level
    pred_err = residual_std_err * np.sqrt(1 + (1/n) + ((log_x_pred - np.mean(log_x))**2 / np.sum((log_x - np.mean(log_x))**2)))

    # Compute upper and lower prediction bands
    upper_pred = 10**(log_y_pred + t_value * pred_err)
    lower_pred = 10**(log_y_pred - t_value * pred_err)

    # Plot the regression line
    ax.plot(x_vals, y_vals, color+'-', lw=1, label="Linear fit")

    # Plot the 95% prediction band
    ax.fill_between(x_vals, lower_pred, upper_pred, color=color, alpha=0.1, label="95\% prediction band")

## All matchups

In [ ]:
## All close
delta=2

ACS_M = {}
HPLC_M = {}
for pft in params["PFT"]:
    acs_values=[]
    hplc_values=[]
    for t in HPLC.time.values:
        matchup = ACS.sel(time=slice(t - pd.Timedelta(minutes=delta), t + pd.Timedelta(minutes=delta)))
        acs_values.append(matchup[pft].values)
        hplc_values.append([HPLC[pft].sel(time=t).values]*matchup[pft].values.shape[0])

    ACS_M[pft] = np.concatenate(acs_values)
    HPLC_M[pft] = np.concatenate(hplc_values)

In [ ]:
fig, ax = plt.subplots(2,3, figsize=(9,6), sharex=True, sharey=True, constrained_layout=True)
# fig.subplots_adjust(hspace=0.01, wspace=0.01)
ax=ax.flatten()
fig.supxlabel('HPLC-derived Chla conc. $[mg\cdot m^{-3}]$')
fig.supylabel('ACS-derived Chla conc. $[mg\cdot m^{-3}]$')

for i, p in enumerate(params["PFT"]):
    ax[i].set_title(p, fontsize=14)
    positive_indices = (HPLC_M[p] > 0) & (ACS_M[p] > 0)
    HPLC_M[p] = HPLC_M[p][positive_indices]
    ACS_M[p] = ACS_M[p][positive_indices]
    
    # Perform the log-log linear fit and get R-squared
    slope, intercept, r_value, p_value, std_err = linregress(np.log10(HPLC_M[p]), np.log10(ACS_M[p]))
    r_squared = r_value**2
    
    # Scatter plot of the data
    ax[i].scatter(x=HPLC_M[p], y=ACS_M[p], c='k', s=1)
    
    # Plot the log-linear line
    x_vals = np.logspace(np.log10(HPLC_M[p].min()), np.log10(HPLC_M[p].max()), 100)
    y_vals = 10**(slope * np.log10(x_vals) + intercept)
    ax[i].plot(x_vals, y_vals, 'r-', lw=1)  # Red line with thickness 2
    
    # Display R^2 on the plot
    ax[i].text(0.02, 0.93, f'$\log(y) = {slope:.2f} \log(x) {intercept:.2f}$', transform=ax[i].transAxes, color='black', fontsize=12)
    ax[i].text(0.02, 0.83, f'$R^2 = {r_squared:.2f}$', transform=ax[i].transAxes, color='black', fontsize=12)
    ax[i].text(0.02, 0.73, f'$Nr.\ {ACS_M[p].size}$',transform=ax[i].transAxes, color='black', fontsize=12)
    
    # Set log scales
    ax[i].set_xscale("log")
    ax[i].set_yscale("log")
    
    # Set axis limits
    ax[i].set_xlim(0.001, 10)
    ax[i].set_ylim(0.001, 10)
    ax[i].grid(alpha=0.3)
    ax[i].plot([0.001, 10], [0.001, 10], 'k-', lw=0.4, alpha=0.3)
    
    ax[i].set_xticks([0.001,0.01,0.1,1,10],[0.001,0.01,0.1,1,10], rotation=30)
    ax[i].set_yticks([0.001,0.01,0.1,1,10],[0.001,0.01,0.1,1,10], rotation=30)
# plt.savefig('fig/PS113_ACS_HPLC_matchup_validation_5min.png', dpi=300, bbox_inches='tight')

## Nearest matchup

In [ ]:
## closest

ACS_M = {}
HPLC_M = {}
for pft in params["PFT"]:
    acs_values=[]
    hplc_values=[]
    for t in HPLC.time.values:
        matchup = ACS[pft].sel(time=t, method='nearest')
        acs_values.append(matchup.values)
        hplc_values.append(HPLC[pft].sel(time=t).values)
    
    ACS_M[pft] = np.array(acs_values)
    HPLC_M[pft] = np.array(hplc_values)

In [ ]:
fig, ax = plt.subplots(2,3, figsize=(9,6), sharex=True, sharey=True, constrained_layout=True)
# fig.subplots_adjust(hspace=0.01, wspace=0.01)
ax=ax.flatten()
fig.supxlabel('HPLC-derived Chla conc. $[mg\cdot m^{-3}]$')
fig.supylabel('ACS-derived Chla conc. $[mg\cdot m^{-3}]$')

for i, p in enumerate(params["PFT"]):
    ax[i].set_title(p, fontsize=14)
    positive_indices = (HPLC_M[p] > 0) & (ACS_M[p] > 0)
    HPLC_M[p] = HPLC_M[p][positive_indices]
    ACS_M[p] = ACS_M[p][positive_indices]
    
    # Perform the log-log linear fit and get R-squared
    slope, intercept, r_value, p_value, std_err = linregress(np.log10(HPLC_M[p]), np.log10(ACS_M[p]))
    r_squared = r_value**2
    
    # Scatter plot of the data
    ax[i].scatter(x=HPLC_M[p], y=ACS_M[p], c='k', s=1)
    
    # Plot the log-linear line
    x_vals = np.logspace(np.log10(HPLC_M[p].min()), np.log10(HPLC_M[p].max()), 100)
    y_vals = 10**(slope * np.log10(x_vals) + intercept)
    ax[i].plot(x_vals, y_vals, 'r-', lw=1)  # Red line with thickness 2
    
    # Display R^2 on the plot
    ax[i].text(0.02, 0.93, f'$\log(y) = {slope:.2f} \log(x) {intercept:.2f}$', transform=ax[i].transAxes, color='black', fontsize=12)
    ax[i].text(0.02, 0.83, f'$R^2 = {r_squared:.2f}$', transform=ax[i].transAxes, color='black', fontsize=12)
    ax[i].text(0.02, 0.73, f'$Nr.\ {ACS_M[p].size}$',transform=ax[i].transAxes, color='black', fontsize=12)
    
    # Set log scales
    ax[i].set_xscale("log")
    ax[i].set_yscale("log")
    
    # Set axis limits
    ax[i].set_xlim(0.001, 10)
    ax[i].set_ylim(0.001, 10)
    ax[i].grid(alpha=0.3)
    ax[i].plot([0.001, 10], [0.001, 10], 'k-', lw=0.4, alpha=0.3)
    
    ax[i].set_xticks([0.001,0.01,0.1,1,10],[0.001,0.01,0.1,1,10], rotation=30)
    ax[i].set_yticks([0.001,0.01,0.1,1,10],[0.001,0.01,0.1,1,10], rotation=30)
# plt.savefig('fig/PS113_ACS_HPLC_matchup_validation_nearest.png', dpi=300, bbox_inches='tight')

## Mean matchup

In [ ]:
## mean close
delta=2

ACS_M = {}
HPLC_M = {}
for pft in params["PFT"]:
    acs_values=[]
    hplc_values=[]
    for timestamp in HPLC.time.values:
        matchup = ACS.sel(time=slice(timestamp - pd.Timedelta(minutes=delta), timestamp + pd.Timedelta(minutes=delta)))
        acs_values.append(matchup.mean(skipna=True)[pft].values)
        hplc_values.append([HPLC[pft].sel(time=timestamp).values])

    ACS_M[pft] = np.array(acs_values)
    HPLC_M[pft] = np.concatenate(hplc_values)

In [ ]:
# Main Plotting Code
fig, ax = plt.subplots(2,3, figsize=(9,6), sharex=True, sharey=True, constrained_layout=True)
ax = ax.flatten()
fig.supxlabel('HPLC-derived Chla conc. $[mg\cdot m^{-3}]$')
fig.supylabel('ACS-derived Chla conc. $[mg\cdot m^{-3}]$')

for i, p in enumerate(params["PFT"]):
    ax[i].set_title(p, fontsize=14)
    positive_indices = (HPLC_M[p] > 0) & (ACS_M[p] > 0) & (np.isfinite(ACS_M[p]))
    HPLC_M[p] = HPLC_M[p][positive_indices]
    ACS_M[p] = ACS_M[p][positive_indices]
    
    # Perform log-log regression
    log_x = np.log10(HPLC_M[p])
    log_y = np.log10(ACS_M[p])
    slope, intercept, r_value, p_value, std_err = linregress(log_x, log_y)
    r_squared = r_value**2

    # Scatter plot
    ax[i].scatter(HPLC_M[p], ACS_M[p], c='k', s=1)
    
    # Plot the prediction band
    plot_prediction_band(ax[i], HPLC_M[p], ACS_M[p], slope, intercept)

    # Display regression equation & R²
    ax[i].text(0.02, 0.93, f'$\log(y) = {slope:.2f} \log(x) {intercept:.2f}$', transform=ax[i].transAxes, color='black', fontsize=12)
    ax[i].text(0.02, 0.83, f'$R^2 = {r_squared:.2f}$', transform=ax[i].transAxes, color='black', fontsize=12)
    ax[i].text(0.02, 0.73, f'$Nr.\ {ACS_M[p].size}$', transform=ax[i].transAxes, color='black', fontsize=12)
    
    # Log scale settings
    ax[i].set_xscale("log")
    ax[i].set_yscale("log")
    
    # Axis limits & grid
    ax[i].set_xlim(0.001, 10)
    ax[i].set_ylim(0.001, 10)
    ax[i].grid(alpha=0.3)
    ax[i].plot([0.001, 10], [0.001, 10], 'k-', lw=0.4, alpha=0.3)
    
    ax[i].set_xticks([0.001,0.01,0.1,1,10], [0.001,0.01,0.1,1,10], rotation=30)
    ax[i].set_yticks([0.001,0.01,0.1,1,10], [0.001,0.01,0.1,1,10], rotation=30)

plt.legend()
# plt.savefig('fig/PS113_ACS_HPLC_matchup_validation_5min_mean.png', dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(6, 1, figsize=(6,9), constrained_layout=True, sharex=True)
fig.supylabel(r'Chla concentration $[mg\cdot m^{-3}]$',fontsize=18)


for idx, pft in enumerate(PFT):
    acs_ds[pft].plot.scatter(ax=ax[idx], s=3, edgecolor='none', c='k', label='Chla ACS',rasterized=True)
    
    hplc_da.sel(PFT=pft).plot.scatter(ax=ax[idx], s=5, edgecolor = 'none', c= 'r', label='Chla HPLC',rasterized=True)
    
    # ax[0].legend(loc='upper center', markerscale=2);
    ax[idx].set_title('')
    ax[idx].set_xlabel('')
    ax[idx].set_ylabel(pft)
    ax[idx].grid(alpha=0.5)
    ax[idx].set_yscale('log')
    ax[idx].set_yticks([0.001,0.01,0.1,1,10], [0.001,0.01,0.1,1,10]);
ax[idx].set_xlabel('Date', fontsize=18)
ax[idx].legend()
# plt.savefig("fig/chla_conc_ACS_and_HPLC.svg", format='svg',dpi=96, bbox_inches='tight', transparent=True)
# plt.savefig("fig/chla_conc_ACS_and_HPLC.pdf", format='pdf',dpi=300, bbox_inches='tight', transparent=True)
# plt.savefig("fig/chla_conc_ACS_and_HPLC.png",dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
# Main Plotting Code
# fig, ax = plt.subplots(6,2, figsize=(9,6), sharex=True, sharey=True, constrained_layout=True, )

fig, ax = plt.subplots(6,2, figsize=(7, 12), gridspec_kw={'width_ratios': [2, 1]}, sharex='col', sharey=True, constrained_layout=True) 
# ax = ax.flatten()

for idx, pft in enumerate(params["PFT"]):
    ##Scatter-----------------------------------------------
    ACS[pft].plot.scatter(x='time', ax=ax[idx,0], s=3, edgecolor='none', c='k', label='Chla ACS',rasterized=True)
    HPLC[pft].plot.scatter(x='time', ax=ax[idx,0], s=5, edgecolor = 'none', c= 'r', label='Chla HPLC',rasterized=True)
    ax[idx,0].set_title('')
    ax[idx,0].set_xlabel('')
    ax[idx,0].set_ylabel(pft)
    ax[idx,0].grid(alpha=0.3)
    ax[idx,0].set_yscale('log')
    ax[idx,0].set_yticks([0.001,0.01,0.1,1,10], [0.001,0.01,0.1,1,10]);
    
    ##Validation------------------------------------------------
    positive_indices = (HPLC_M[pft] > 0) & (ACS_M[pft] > 0) & (np.isfinite(ACS_M[pft]))
    HPLC_M[pft] = HPLC_M[pft][positive_indices]
    ACS_M[pft] = ACS_M[pft][positive_indices]
    
    # Perform log-log regression
    log_x = np.log10(HPLC_M[pft])
    log_y = np.log10(ACS_M[pft])
    slope, intercept, r_value, p_value, std_err = linregress(log_x, log_y)
    r_squared = r_value**2

    # Scatter plot
    ax[idx,1].scatter(HPLC_M[pft], ACS_M[pft], c='k', s=1)
    
    # Plot the prediction band
    plot_prediction_band(ax[idx,1], HPLC_M[pft], ACS_M[pft], slope, intercept)

    # Display regression equation & R²
    ax[idx,1].text(0.02, 0.93, f'$\log(y) = {slope:.2f} \log(x) {intercept:.2f}$', transform=ax[idx,1].transAxes, color='black', fontsize=10)
    ax[idx,1].text(0.02, 0.83, f'$R^2 = {r_squared:.2f}$', transform=ax[idx,1].transAxes, color='black', fontsize=10)
    ax[idx,1].text(0.02, 0.73, f'$Nr.\ {ACS_M[pft].size}$', transform=ax[idx,1].transAxes, color='black', fontsize=10)
    
    # Log scale settings
    ax[idx,1].set_xscale("log")
    ax[idx,1].set_yscale("log")
    
    # Axis limits & grid
    ax[idx,1].set_xlim(0.001, 10)
    ax[idx,1].set_ylim(0.001, 10)
    ax[idx,1].grid(alpha=0.3)
    ax[idx,1].plot([0.001, 10], [0.001, 10], 'k-', lw=0.4, alpha=0.3)
    
    ax[idx,1].set_xticks([0.001,0.01,0.1,1,10], [0.001,0.01,0.1,1,10])
    ax[idx,1].set_yticks([0.001,0.01,0.1,1,10], [0.001,0.01,0.1,1,10])
    # ax[idx,1].spines['bottom'].set_color('red')
    # ax[idx,1].tick_params(axis='x', colors='red', which='both')
    # ax[idx,1].spines['left'].set_color('blue')
    # ax[idx,1].tick_params(axis='y', colors='blue', which='both', labelleft=True)
    
ax[idx,0].set_xlabel('Date', fontsize=18)
ax[idx,0].legend()
ax[idx,1].legend(loc='lower right')
ax[idx,1].set_xlabel('HPLC-derived Chla conc.'+'\n'+'$[mg\cdot m^{-3}]$', fontsize=12, fontweight='bold');

# ax[:,1].set_ylabel('ACS-derived Chla conc. $[mg\cdot m^{-3}]$')


# plt.savefig('fig/PS113_ACS_HPLC_matchup_validation_5min_mean_validation.png', dpi=300, bbox_inches='tight')